In [113]:
import numpy as np
import scipy as sp
import sympy as smp
from sympy import kronecker_product as kron
from sympy import init_printing, print_latex
import random

The equations for $Ann_\mathfrak{g}(\psi \otimes 0)$ are: $$ \alpha a + \beta b = - \alpha e \\ \alpha c - \beta a = - \beta e \\ f = 0 \\ d \in \mathbb{C}^2 $$

In [114]:
α, β = smp.symbols("α β")

# basis for sl2
e = smp.Matrix([ # raise
    [0, 1],
    [0, 0]
])
f = smp.Matrix([ # lower
    [0, 0],
    [1, 0]
])
h = smp.Matrix([ # cartan
    [1, 0],
    [0, -1]
])

I = smp.eye(2)
Z = smp.zeros(2)

# basis for sl2 + sl2
# pick the order so that sympy uses the same free variables I chose (last 3 are cHOSEN TO BE FREE)

basis = [
    (f, Z),   # c  -> determined
    (Z, h),   # e  -> determined
    (Z, f),   # f  -> determined
    (h, Z),   # a  -> free
    (e, Z),   # b  -> free
    (Z, e),   # d  -> free
]
 
def rho(A, B):
    return kron(A, I) + kron(I, B)

sl2_2_basis = [rho(A,B) for (A, B) in basis]

psi = smp.Matrix([
    [α],
    [β]
])
ket0 = smp.Matrix([
    [1],
    [0]
])

ket_psi_0 = kron(psi, ket0)

display(len(sl2_2_basis))
display(sl2_2_basis[0])
display(kron(I, e))

6

⎡0  0  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢1  0  0  0⎥
⎢          ⎥
⎣0  1  0  0⎦

⎡0  1  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  1⎥
⎢          ⎥
⎣0  0  0  0⎦

In [115]:
# get the matrix of all actionsso 
M = smp.Matrix.hstack(*[g * ket_psi_0 for g in sl2_2_basis])
M

# with the two qubit basis |00>, |01>, |10>, |11>, the columns are 

⎡0  α  0  α   β  0⎤
⎢                 ⎥
⎢0  0  α  0   0  0⎥
⎢                 ⎥
⎢α  β  0  -β  0  0⎥
⎢                 ⎥
⎣0  0  β  0   0  0⎦

In [116]:
# take the nullspace of this matrix:

nullspace = M.nullspace()

for m in nullspace:
    display(m)

⎡2⋅β⎤
⎢───⎥
⎢ α ⎥
⎢   ⎥
⎢-1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎣ 0 ⎦

⎡ 2 ⎤
⎢β  ⎥
⎢── ⎥
⎢ 2 ⎥
⎢α  ⎥
⎢   ⎥
⎢-β ⎥
⎢───⎥
⎢ α ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 1 ⎥
⎢   ⎥
⎣ 0 ⎦

⎡0⎤
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎣1⎦

In [117]:
M.rank(), len(nullspace)

(3, 3)

In [152]:
# the nullspace of M is just vectors Mv = 0, where v = [b, c, a, d, f, e], aka a chosen basis for Ann_g()
# reshape the matrices to get the proper basis
ann_g_basis = []

for k, n in enumerate(nullspace):
    A = smp.simplify(sum((n[i] * basis[i][0] for i in range(6)), smp.zeros(2)))
    B = smp.simplify(sum((n[i] * basis[i][1] for i in range(6)), smp.zeros(2)))
    
    
    print(f'\nbasis vector {k+1}:')
    display((A, B))
    rep = kron(A, I) + kron(I, B)
    display(rep)
    ann_g_basis.append(rep)

random_c = lambda: random.random() + 1j * random.random()
def random_c2():
    vec = [random_c(), random_c()]
    return vec / np.linalg.norm(vec)

print(f"Random psi in C2: \n{random_c2()}")

# show that for a random vector in the annihilator, it always annihilates
rep = sum(random_c() * np.array(X.tolist()) for X in ann_g_basis)

print(f"Random matrix in Ann_g(|ψ0>): {rep}")

# evaluate for a random |ψ>
(rep @ ket_psi_0).subs({α: random_c2()[0], β: random_c2()[1]}).evalf(2)


basis vector 1:


⎛⎡ 1   0 ⎤         ⎞
⎜⎢       ⎥  ⎡-1  0⎤⎟
⎜⎢2⋅β    ⎥, ⎢     ⎥⎟
⎜⎢───  -1⎥  ⎣0   1⎦⎟
⎝⎣ α     ⎦         ⎠

⎡ 0    0   0   0⎤
⎢               ⎥
⎢ 0    2   0   0⎥
⎢               ⎥
⎢2⋅β            ⎥
⎢───   0   -2  0⎥
⎢ α             ⎥
⎢               ⎥
⎢     2⋅β       ⎥
⎢ 0   ───  0   0⎥
⎣      α        ⎦


basis vector 2:


⎛⎡0   1⎤  ⎡-β    ⎤⎞
⎜⎢     ⎥  ⎢───  0⎥⎟
⎜⎢ 2   ⎥  ⎢ α    ⎥⎟
⎜⎢β    ⎥, ⎢      ⎥⎟
⎜⎢──  0⎥  ⎢     β⎥⎟
⎜⎢ 2   ⎥  ⎢ 0   ─⎥⎟
⎝⎣α    ⎦  ⎣     α⎦⎠

...


basis vector 3:


⎛⎡0  0⎤  ⎡0  1⎤⎞
⎜⎢    ⎥, ⎢    ⎥⎟
⎝⎣0  0⎦  ⎣0  0⎦⎠

⎡0  1  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  1⎥
⎢          ⎥
⎣0  0  0  0⎦

Random psi in C2: 
[0.23509423+0.6038191j  0.64479139+0.40543465j]
Random matrix in Ann_g(|ψ0>): [[-β*(0.855153350936118 + 0.163545710934982*I)/α
  0.361947479414648 + 0.188454713949843*I
  0.855153350936118 + 0.163545710934982*I 0]
 [0
  1.43767259964912 + 0.564163512951164*I + β*(0.855153350936118 + 0.163545710934982*I)/α
  0 0.855153350936118 + 0.163545710934982*I]
 [2*β*(0.718836299824562 + 0.282081756475582*I)/α + β**2*(0.855153350936118 + 0.163545710934982*I)/α**2
  0
  -1.43767259964912 - 0.564163512951164*I - β*(0.855153350936118 + 0.163545710934982*I)/α
  0.361947479414648 + 0.188454713949843*I]
 [0
  2*β*(0.718836299824562 + 0.282081756475582*I)/α + β**2*(0.855153350936118 + 0.163545710934982*I)/α**2
  0 β*(0.855153350936118 + 0.163545710934982*I)/α]]


⎡         0          ⎤
⎢                    ⎥
⎢         0          ⎥
⎢                    ⎥
⎢-1.2e-17 - 1.7e-17⋅ⅈ⎥
⎢                    ⎥
⎣         0          ⎦

In [ ]:
# now lets move onto computing the complement of this annihilator to get the subalgebra that doesn't take the vector to 0

